In [25]:
import tensorflow as tf 
from tensorflow.keras.models import load_model
import pickle 
import pandas as pd
import numpy as np

In [26]:
#  Load the trained model , scaler pickle , onehot encoder pickle and the test data
model = load_model('model.h5')

# load the encoder and scaler pickle files
with open('onehot_encoder_geo.pkl', 'rb') as file:
    onehot_encoder_geo = pickle.load(file)

with open('label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)    

with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)


In [27]:
# example test data for prediction
test_data ={
    'CreditScore': 700,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [28]:
#  one-hot encode the 'Geography' column using the loaded encoder
geo_encoded = onehot_encoder_geo.transform([[test_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

geo_encoded_df


e:\AIML\DeepLearning\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [29]:
input_data = pd.DataFrame([test_data])
input_data 

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,700,France,Male,40,3,60000,2,1,1,50000


In [30]:
# combine one hot encoded columns with the input data\
input_data = pd.concat([input_data.reset_index(drop=True) , geo_encoded_df], axis=1)
input_data 

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,700,France,Male,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [31]:
#  Encode the categorical variables 
input_data['Gender'] = label_encoder_gender.transform(input_data['Gender']) 
input_data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,700,France,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [42]:
#  Scaling the data 
input_data = input_data[scaler.feature_names_in_]

input_scaled = scaler.transform(input_data)

print(input_scaled)

[[ 0.50178794  0.91324755  0.10479359 -0.69539349 -0.25781119  0.80843615
   0.64920267  0.97481699 -0.87683221  1.00150113 -0.57946723 -0.57638802]]


In [36]:
print(scaler)
print(scaler.mean_)
print(scaler.scale_)

StandardScaler()
[6.51647625e+02 5.45250000e-01 3.88977500e+01 5.00387500e+00
 7.61021396e+04 1.53137500e+00 7.03500000e-01 5.12750000e-01
 1.00431290e+05 4.99250000e-01 2.51375000e-01 2.49375000e-01]
[9.63601770e+01 4.97948228e-01 1.05182957e+01 2.88164189e+00
 6.24571018e+04 5.79668534e-01 4.56714079e-01 4.99837411e-01
 5.75153254e+04 4.99999437e-01 4.33803653e-01 4.32651256e-01]


In [40]:
print(scaler.feature_names_in_)


['CreditScore' 'Gender' 'Age' 'Tenure' 'Balance' 'NumOfProducts'
 'HasCrCard' 'IsActiveMember' 'EstimatedSalary' 'Geography_France'
 'Geography_Germany' 'Geography_Spain']


In [41]:
print(input_data.columns)

Index(['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance',
       'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary',
       'Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype='str')


In [46]:
# Make prediction using the trained model 

prediction = model.predict(input_scaled)
prdediction

1/1 [==============================] - 0s 36ms/step


array([[0.01793833]], dtype=float32)

In [47]:
prediction_proba= prediction[0][0] 

In [48]:
prediction_proba

0.017938325

In [49]:
if prediction_proba > 0.5:
    print("The customer is likely to churn.")
else :
    print("The customer is not likely to churn.")    

The customer is not likely to churn.
